In [ ]:
%pip install boto3

In [ ]:
from pypdf import PdfReader, PdfWriter
import boto3
import json

In [ ]:

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"   # o tu región
)

response = bedrock.invoke_model(
    modelId="amazon.titan-embed-text-v2:0",
    body=json.dumps({
        "inputText": "¿Qué descuentos ofrece Cibertec a miembros de la PNP?"
    })
)

result = json.loads(response["body"].read())

embedding = result["embedding"]

print(len(embedding))
print(embedding[:5])

In [ ]:
pdf_knowledge=["./data/CONVENIOS INSTITUCIONES.pdf", "./data/RTB'S CIBERTEC.pdf"]

merger = PdfWriter()

for pdf_path in pdf_knowledge:
    merger.append(pdf_path)

In [ ]:
merged_pdf_path = "./data/KNOWLEDGE.pdf"
with open(merged_pdf_path, "wb") as f_out:
    merger.write(f_out)
    merger.close()

In [ ]:
reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        text += extracted + "\n"

print(f"PDF concatenado guardado en :  {merged_pdf_path}")
print(f"Texto total extraido del PDF combinado {len(text)} caracteres")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_text(text)

In [ ]:
vectors = []

for chunk in chunks:

    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({
            "inputText": chunk
        })
    )

    embedding = json.loads(
        response["body"].read()
    )["embedding"]

    vectors.append({
        "text": chunk,
        "embedding": embedding
    })

In [ ]:
import os
from pinecone import Pinecone
from dotenv import load_dotenv

# 1. Cargar variables de entorno (por si usas un archivo .env para tu API Key)
load_dotenv()

# 2. Configurar la API key y el nombre del índice
# Nota: Puedes reemplazar esto directamente con tus strings si no usas .env
pinecone_api_key = os.environ.get("PINECONE_API_KEY_AWS")
index_name = "poc-ia-prospecter"  # Ej: "cibertec-index"

# 3. Inicializar el cliente de Pinecone
pc = Pinecone(api_key=pinecone_api_key)

# 4. Conectarse al índice
index = pc.Index(index_name)


In [ ]:
index.upsert(
    vectors=[
        (
            f"chunk-{i}",
            item["embedding"],
            {"text": item["text"]}
        )
        for i, item in enumerate(vectors)
    ]
)


In [ ]:
import boto3

# 1. Crear el cliente de s3vectors en la región us-east-2
s3_vectors_client = boto3.client('s3vectors', region_name='us-east-2')

# 2. El ARN de tu nuevo índice vectorial de 1024 dimensiones
index_arn_1024 = 'arn:aws:s3vectors:us-east-2:084828568567:bucket/poc-ia-prospecter/index/index-data-test'

# 3. Transformar tus vectores de 1024 dimensiones ya existentes al formato S3 Vectors
s3_vectors = [
    {
        "key": f"chunk-{i}",
        "data": {
            "float32": item["embedding"]
        },
        "metadata": {
            "text": item["text"]
        }
    }
    for i, item in enumerate(vectors)
]

# 4. Realizar la subida a S3
print(f"Subiendo {len(s3_vectors)} vectores a tu nuevo índice en S3...")
response_s3 = s3_vectors_client.put_vectors(
    indexArn=index_arn_1024,
    vectors=s3_vectors
)
print("¡Subida completada con éxito!")
print(response_s3)


In [ ]:


s3_vectors_client = boto3.client('s3vectors', region_name='us-east-2')

# Consultar la lista de vectores en el índice
response = s3_vectors_client.list_vectors(
    vectorBucketName='poc-ia-prospecter',
    indexName='index-data-test'
)

# Imprimir las llaves para verificar
for vector in response.get('vectors', []):
    print(f"Vector cargado: {vector['key']}")
